# Label Smoothing — Rethinking the Inception Architecture (2015)

_Papers · Foundations & Optimization_

**Stop training the network to be 100% certain: replace the hard one-hot target with a softened target that keeps a little probability mass on every wrong class, which curbs over-confident logits and improves calibration.**

---

This notebook is a practice scaffold for the **Label Smoothing — Rethinking the Inception Architecture (2015)** lesson. We build label smoothing from scratch, check it against PyTorch's built-in version, then run a small ablation that makes the calibration effect visible — one step at a time. Run the cells, read the note above each, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## Reference implementation — PyTorch

### Step 1 — Build smoothed cross-entropy and check it against PyTorch

Label smoothing replaces the one-hot target $\delta_{k,y}$ with a softened target $q'(k) = (1-\epsilon)\,\delta_{k,y} + \epsilon/K$ — it shaves $\epsilon$ off the true class and sprinkles it uniformly over all $K$ classes. The loss is then the cross-entropy between this soft target and the model's log-probabilities.

To prove our from-scratch version is correct, we check it equals PyTorch's `nn.CrossEntropyLoss(label_smoothing=eps)` on random logits — that built-in is our oracle.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)

# Label smoothing from scratch (Section 7 of Szegedy et al., 2015, uniform case).
def my_smoothed_ce(logits, targets, eps):
    K = logits.size(1)
    onehot = F.one_hot(targets, K).float()        # delta_{k,y}
    smooth = (1.0 - eps) * onehot + eps / K       # q'(k) = (1-eps)*onehot + eps/K
    logp = F.log_softmax(logits, dim=1)           # stable log-softmax
    return -(smooth * logp).sum(dim=1).mean()     # H(q', p), averaged over batch

# THE ORACLE: must equal nn.CrossEntropyLoss(label_smoothing=eps).
N, K, eps = 8, 5, 0.1
logits = torch.randn(N, K)
targets = torch.randint(0, K, (N,))

mine = my_smoothed_ce(logits, targets, eps)
torchs = nn.CrossEntropyLoss(label_smoothing=eps)(logits, targets)
print("mine   :", mine.item())
print("torch  :", torchs.item())
print("allclose(mine, nn.CrossEntropyLoss(label_smoothing)):",
      torch.allclose(mine, torchs, atol=1e-6))    # expect True

### Step 2 — Recompute the worked example by hand and in code

Here is the small example from the lesson: $K=5$ classes, true label $y=2$, smoothing $\epsilon=0.1$. The smoothed target becomes $[0.02, 0.02, 0.92, 0.02, 0.02]$ — the true class drops from $1.0$ to $0.92$ and each other class rises from $0$ to $0.02$.

Notice the smoothed loss ($\approx 0.47$) is *higher* than plain cross-entropy ($\approx 0.25$): smoothing always adds a floor of loss, which is precisely what stops the model from chasing 100% confidence.

In [ ]:
# Recompute the worked example: K=5, true label y=2, eps=0.1.
z = torch.tensor([[1., 0., 3., 0., 0.]])
y = torch.tensor([2])

onehot = F.one_hot(y, 5).float()
qprime = (1 - 0.1) * onehot + 0.1 / 5
print("smoothed target:", qprime.tolist())        # [[0.02,0.02,0.92,0.02,0.02]]

plain_ce = F.cross_entropy(z, y).item()
smoothed_ce = my_smoothed_ce(z, y, 0.1).item()
print("plain CE  (eps=0)   :", plain_ce)            # ~0.2505
print("smoothed  (eps=0.1) :", smoothed_ce)         # ~0.4705

### Step 3 — Set up a hard toy problem and a training routine

To *see* the regularizing effect we need a task hard enough that confidence and accuracy can diverge. We make three overlapping Gaussian blobs (noise $1.6$), so the classes genuinely overlap and a model can be confidently wrong.

The `train` function fits the same tiny MLP from an identical initialization, varying only `label_smoothing`. Afterward it measures test accuracy, mean predicted-class confidence, and the confidence-minus-accuracy gap — a simple proxy for miscalibration.

In [ ]:
# 3-class toy data: three OVERLAPPING Gaussian blobs (noise=1.6 makes the task hard
# enough that confidence and accuracy can diverge -> over-confidence is visible).
torch.manual_seed(1)

def make_data(n):
    centers = torch.tensor([[1.2, 1.2], [-1.2, 1.2], [0., -1.2]])
    ys = torch.randint(0, 3, (n,))
    X = centers[ys] + 1.6 * torch.randn(n, 2)
    return X, ys

Xtr, ytr = make_data(600)
Xte, yte = make_data(600)

def train(eps, steps=600):
    torch.manual_seed(2)                          # identical init for both
    net = nn.Sequential(nn.Linear(2, 64), nn.ReLU(), nn.Linear(64, 3))
    opt = torch.optim.Adam(net.parameters(), lr=0.03)
    lossf = nn.CrossEntropyLoss(label_smoothing=eps)
    for _ in range(steps):
        opt.zero_grad()
        loss = lossf(net(Xtr), ytr)
        loss.backward()
        opt.step()
    with torch.no_grad():
        p = F.softmax(net(Xte), dim=1)
        conf, pred = p.max(dim=1)
        acc = (pred == yte).float().mean().item()
        mean_conf = conf.mean().item()
        ece = (mean_conf - acc)                    # simple confidence-minus-accuracy gap
    return acc, mean_conf, ece

### Step 4 — Run the ablation: one-hot vs smoothed

Now we train twice — $\epsilon=0$ (plain one-hot) and $\epsilon=0.1$ — and compare. Expect accuracy to stay essentially the same, but the smoothed model to report *lower* mean confidence and a *smaller* confidence-accuracy gap: better calibrated, not more accurate.

In [ ]:
for e in [0.0, 0.1]:
    acc, mc, ece = train(e)
    print(f"eps={e}: test acc {acc:.3f}, mean confidence {mc:.3f}, conf-acc gap {ece:+.3f}")
# expect (similar accuracy, but different confidence):
#   eps=0.0: acc ~0.635, mean conf ~0.711, gap +0.076  (over-confident)
#   eps=0.1: acc ~0.637, mean conf ~0.669, gap +0.032  (better calibrated)

## Visualize the data & results

_Does label smoothing (eps=0.1) curb over-confident logits and improve calibration versus one-hot (eps=0), at similar accuracy — on a toy 3-class problem with everything else held fixed?_

### Step 1 — Rebuild the toy data and training routine standalone

This visualization block is self-contained, so we re-import torch and redefine the overlapping-blobs data plus the `train` routine. Keeping it standalone means you can run this section on its own without first running the reference implementation.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(1)

def make_data(n):                                 # overlapping blobs -> hard task
    centers = torch.tensor([[1.2, 1.2], [-1.2, 1.2], [0., -1.2]])
    ys = torch.randint(0, 3, (n,))
    X = centers[ys] + 1.6 * torch.randn(n, 2)
    return X, ys

Xtr, ytr = make_data(600)
Xte, yte = make_data(600)

def train(eps, steps=600):
    torch.manual_seed(2)                          # identical init
    net = nn.Sequential(nn.Linear(2, 64), nn.ReLU(), nn.Linear(64, 3))
    opt = torch.optim.Adam(net.parameters(), lr=0.03)
    lossf = nn.CrossEntropyLoss(label_smoothing=eps)
    for _ in range(steps):
        opt.zero_grad()
        loss = lossf(net(Xtr), ytr)
        loss.backward()
        opt.step()
    with torch.no_grad():
        conf, pred = F.softmax(net(Xte), 1).max(1)
        acc = (pred == yte).float().mean().item()
        mean_conf = conf.mean().item()
        gap = mean_conf - acc
        return round(acc, 3), round(mean_conf, 3), round(gap, 3)

### Step 2 — Compare confidence and calibration across the two runs

We train at $\epsilon=0$ and $\epsilon=0.1$ and print accuracy, mean confidence, and the confidence-accuracy gap for each. The one-hot model lands over-confident (large positive gap); the smoothed model pulls confidence down toward its true accuracy, shrinking the gap — at essentially unchanged accuracy.

In [ ]:
for e in [0.0, 0.1]:
    acc, mc, gap = train(e)
    print(f"eps={e}: acc {acc}, mean_conf {mc}, conf-acc gap {gap:+}")
# eps=0.0: acc 0.635, mean_conf 0.711, gap +0.076   (over-confident)
# eps=0.1: acc 0.637, mean_conf 0.669, gap +0.032   (better calibrated)

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** With $K=4$ and $\epsilon=0.2$, true label $y=0$, write the smoothed target vector and verify it sums to 1.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Compute $\epsilon/K = 0.2/4 = 0.05$. — _Every class — including the true one — gets this added._
- True class: $(1-0.2)\cdot1 + 0.05 = 0.8 + 0.05 = 0.85$. — _Scale the one-hot by $1-\epsilon$, then add $\epsilon/K$._
- Each other class: $0 + 0.05 = 0.05$. — _One-hot is 0 there, so only the sprinkled $\epsilon/K$ remains._

**Answer:** Target $=[0.85,\,0.05,\,0.05,\,0.05]$, which sums to $0.85+3(0.05)=1.0$. Note the true class is $0.85$, not $0.80$.

</details>

**Problem 2.** Why does the smoothed cross-entropy have a finite minimizer in the logits, while plain one-hot cross-entropy does not?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Plain: minimizing $-\log p(y)$ needs $p(y)\to1$, which needs $z_y - z_k \to +\infty$. — _Softmax reaches 1 only in the infinite-logit limit._
- Smoothed: the target assigns $\epsilon/K\gt0$ to every class, so the loss is minimized when $p$ matches $q'$, i.e. $p(y)=1-\epsilon+\epsilon/K\lt1$. — _Matching a soft target requires only finite logits._
- Therefore the gradient stops pushing once $p\approx q'$. — _There is an attainable optimum._

**Answer:** One-hot demands $p(y)=1$, attainable only at infinite logit gaps, so training keeps inflating the true logit — over-confidence. The smoothed target's optimum is $p(y)=1-\epsilon+\epsilon/K\lt1$, a finite logit configuration, so the model settles at bounded confidence. That bounded optimum is exactly the regularizing effect.

</details>

**Problem 3.** Ablation: train the same tiny classifier twice — once with $\epsilon=0$ (one-hot) and once with $\epsilon=0.1$ — and compare (a) the mean confidence of the predicted class and (b) calibration on held-out data. What changes, and what does not?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Fix seed, data, architecture and optimizer; vary only the loss's label_smoothing between 0 and 0.1. — _Isolate smoothing as the single variable._
- After training, on a test set record the mean top-class probability (confidence) and the expected calibration error (gap between confidence and accuracy). — _Over-confidence and calibration are the claimed effects._
- Compare: the $\epsilon=0.1$ model should report lower, less extreme confidence and smaller calibration error, at similar accuracy. — _The regularizer caps confidence without necessarily changing the top-1 decision._

**Answer:** In our small run (see CODEVIZ — our numbers, not the paper's) the $\epsilon=0$ model is over-confident — mean predicted-class probability $0.711$ while it is right only $63.5\%$ of the time, a $+0.076$ gap — while $\epsilon=0.1$ pulls mean confidence down to $0.669$, much closer to its $63.7\%$ accuracy (gap $+0.032$). Test accuracy is essentially unchanged. What changed: confidence/calibration. What did not: the architecture, the data, and (largely) the top-1 accuracy — mirroring the paper's small ~0.2% accuracy effect but visible calibration benefit.

</details>